# Chapitre 6 — Structuration et transformation

**Durée estimée : 8-10 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Restructurer** des DataFrames avec pivot et melt selon les besoins d'analyse
2. **Combiner** des sources de données avec merge et concat en choisissant la bonne méthode
3. **Agréger** des données avec groupby et créer des statistiques par groupe
4. **Créer** de nouvelles variables pertinentes (feature engineering) pour l'analyse et le ML

---

## 6.2 Combinaison de données

### Merge : joindre sur une clé

In [1]:
import pandas as pd
import numpy as np

# Création des tables
clients = pd.DataFrame({
    'client_id': [1, 2, 3],
    'nom': ['Alice', 'Bob', 'Charlie'],
    'ville': ['Paris', 'Lyon', 'Marseille']
})

commandes = pd.DataFrame({
    'commande_id': [101, 102, 103, 104],
    'client_id': [1, 1, 2, 4],  # Note: client 4 n'existe pas dans clients
    'montant': [100, 150, 200, 50]
})

print("Table clients :")
print(clients)
print("\nTable commandes :")
print(commandes)

Table clients :
   client_id      nom      ville
0          1    Alice      Paris
1          2      Bob       Lyon
2          3  Charlie  Marseille

Table commandes :
   commande_id  client_id  montant
0          101          1      100
1          102          1      150
2          103          2      200
3          104          4       50


In [2]:
# Inner join (intersection) - seulement ce qui matche
df_inner = pd.merge(clients, commandes, on='client_id', how='inner')
print("INNER JOIN :")
print(df_inner)

INNER JOIN :
   client_id    nom  ville  commande_id  montant
0          1  Alice  Paris          101      100
1          1  Alice  Paris          102      150
2          2    Bob   Lyon          103      200


In [3]:
# Left join (tous les clients)
df_left = pd.merge(clients, commandes, on='client_id', how='left')
print("LEFT JOIN (tous les clients) :")
print(df_left)

LEFT JOIN (tous les clients) :
   client_id      nom      ville  commande_id  montant
0          1    Alice      Paris        101.0    100.0
1          1    Alice      Paris        102.0    150.0
2          2      Bob       Lyon        103.0    200.0
3          3  Charlie  Marseille          NaN      NaN


In [4]:
# Outer join (union)
df_outer = pd.merge(clients, commandes, on='client_id', how='outer')
print("OUTER JOIN (union) :")
print(df_outer)

OUTER JOIN (union) :
   client_id      nom      ville  commande_id  montant
0          1    Alice      Paris        101.0    100.0
1          1    Alice      Paris        102.0    150.0
2          2      Bob       Lyon        103.0    200.0
3          3  Charlie  Marseille          NaN      NaN
4          4      NaN        NaN        104.0     50.0


### Cas courants de merge

In [5]:
# Tables avec noms de colonnes différents
clients_v2 = pd.DataFrame({
    'id_client': [1, 2, 3],
    'nom': ['Alice', 'Bob', 'Charlie']
})

commandes_v2 = pd.DataFrame({
    'customer_id': [1, 1, 2],
    'montant': [100, 150, 200]
})

# Merge avec clés différentes
df_merge = pd.merge(
    clients_v2,
    commandes_v2,
    left_on='id_client',
    right_on='customer_id',
    how='left'
)
print("Merge avec clés différentes :")
print(df_merge)

Merge avec clés différentes :
   id_client      nom  customer_id  montant
0          1    Alice          1.0    100.0
1          1    Alice          1.0    150.0
2          2      Bob          2.0    200.0
3          3  Charlie          NaN      NaN


In [ ]:
# Merge sur plusieurs colonnes probleme 

ventes = pd.DataFrame({
    'annee': [2023, 2023, 2024, 2024],
    'region': ['Nord', 'Sud', 'Nord', 'Sud'],
    'ventes': [1000, 1200, 1100, 1300]
})

objectifs = pd.DataFrame({
    'annee': [2023, 2023, 2024, 2024],
    'region': ['Nord', 'Sud', 'Nord', 'Sud'],
    'objectif': [950, 1100, 1050, 1250]
})

df_merge_multi = pd.merge(ventes, objectifs, on='annee', how='left')
df_merge_multi['ecart'] = df_merge_multi['ventes'] - df_merge_multi['objectif']

print("Merge sur plusieurs colonnes :")
print(df_merge_multi)

Merge sur plusieurs colonnes :
   annee region_x  ventes region_y  objectif  ecart
0   2023     Nord    1000     Nord       950     50
1   2023     Nord    1000      Sud      1100   -100
2   2023      Sud    1200     Nord       950    250
3   2023      Sud    1200      Sud      1100    100
4   2024     Nord    1100     Nord      1050     50
5   2024     Nord    1100      Sud      1250   -150
6   2024      Sud    1300     Nord      1050    250
7   2024      Sud    1300      Sud      1250     50


In [ ]:
# Merge sur plusieurs colonnes solution
ventes = pd.DataFrame({
    'annee': [2023, 2023, 2024, 2024],
    'region': ['Nord', 'Sud', 'Nord', 'Sud'],
    'ventes': [1000, 1200, 1100, 1300]
})

objectifs = pd.DataFrame({
    'annee': [2023, 2023, 2024, 2024],
    'region': ['Nord', 'Sud', 'Nord', 'Sud'],
    'objectif': [950, 1100, 1050, 1250]
})

df_merge_multi = pd.merge(ventes, objectifs, on=['annee', 'region'], how='left')
df_merge_multi['ecart'] = df_merge_multi['ventes'] - df_merge_multi['objectif']

print("Merge sur plusieurs colonnes :")
print(df_merge_multi)

Merge sur plusieurs colonnes :
   annee region  ventes  objectif  ecart
0   2023   Nord    1000       950     50
1   2023    Sud    1200      1100    100
2   2024   Nord    1100      1050     50
3   2024    Sud    1300      1250     50


### ✍️ Exercice 6.3 : Jointures multiples (15 min)

In [8]:
import pandas as pd

# Trois tables à joindre
clients = pd.DataFrame({
    'client_id': [1, 2, 3, 4],
    'nom': ['Alice', 'Bob', 'Charlie', 'David'],
    'segment': ['Premium', 'Standard', 'Premium', 'Standard']
})

commandes = pd.DataFrame({
    'commande_id': [101, 102, 103, 104, 105],
    'client_id': [1, 1, 2, 5, 3],  # client 5 n'existe pas
    'produit_id': [10, 20, 10, 30, 20],
    'quantite': [2, 1, 5, 3, 2]
})

produits = pd.DataFrame({
    'produit_id': [10, 20, 30],
    'nom_produit': ['Widget A', 'Widget B', 'Widget C'],
    'prix': [50, 75, 100]
})

print("Table clients :")
print(clients)
print("\nTable commandes :")
print(commandes)
print("\nTable produits :")
print(produits)

Table clients :
   client_id      nom   segment
0          1    Alice   Premium
1          2      Bob  Standard
2          3  Charlie   Premium
3          4    David  Standard

Table commandes :
   commande_id  client_id  produit_id  quantite
0          101          1          10         2
1          102          1          20         1
2          103          2          10         5
3          104          5          30         3
4          105          3          20         2

Table produits :
   produit_id nom_produit  prix
0          10    Widget A    50
1          20    Widget B    75
2          30    Widget C   100


In [9]:
# Étape 1 : Joindre commandes et clients (left join sur commandes)
df = pd.merge(commandes, clients, on='client_id', how='left')
print("Après jointure commandes + clients :")
print(df)

Après jointure commandes + clients :
   commande_id  client_id  produit_id  quantite      nom   segment
0          101          1          10         2    Alice   Premium
1          102          1          20         1    Alice   Premium
2          103          2          10         5      Bob  Standard
3          104          5          30         3      NaN       NaN
4          105          3          20         2  Charlie   Premium


In [10]:
# Étape 2 : Joindre avec produits
df = pd.merge(df, produits, on='produit_id', how='left')
print("\nAprès jointure avec produits :")
print(df)


Après jointure avec produits :
   commande_id  client_id  produit_id  quantite      nom   segment  \
0          101          1          10         2    Alice   Premium   
1          102          1          20         1    Alice   Premium   
2          103          2          10         5      Bob  Standard   
3          104          5          30         3      NaN       NaN   
4          105          3          20         2  Charlie   Premium   

  nom_produit  prix  
0    Widget A    50  
1    Widget B    75  
2    Widget A    50  
3    Widget C   100  
4    Widget B    75  


In [ ]:
# ou
df = (
    commandes
    .merge(clients, on='client_id', how='left')
    .merge(produits, on='produit_id', how='left')
)

In [11]:
# Étape 3 : Calculer le montant total par commande
df['montant'] = df['quantite'] * df['prix']

# Étape 4 : Afficher les commandes du segment Premium
print("\nCommandes Premium :")
print(df[df['segment'] == 'Premium'])

# Question : La commande 104 (client_id=5) apparaît-elle dans le résultat ?
print("\n→ La commande 104 apparaît mais avec NaN pour le client (client_id=5 n'existe pas)")


Commandes Premium :
   commande_id  client_id  produit_id  quantite      nom  segment nom_produit  \
0          101          1          10         2    Alice  Premium    Widget A   
1          102          1          20         1    Alice  Premium    Widget B   
4          105          3          20         2  Charlie  Premium    Widget B   

   prix  montant  
0    50      100  
1    75       75  
4    75      150  

→ La commande 104 apparaît mais avec NaN pour le client (client_id=5 n'existe pas)


### Concat : empiler des DataFrames

In [12]:
# Empiler verticalement (ajout de lignes)
df_jan = pd.DataFrame({'mois': ['Jan']*3, 'ventes': [100, 120, 90]})
df_fev = pd.DataFrame({'mois': ['Fev']*3, 'ventes': [110, 130, 95]})
df_mar = pd.DataFrame({'mois': ['Mar']*3, 'ventes': [105, 125, 100]})

df_q1 = pd.concat([df_jan, df_fev, df_mar], ignore_index=True)
print("Concat vertical (ignore_index=True) :")
print(df_q1)

Concat vertical (ignore_index=True) :
  mois  ventes
0  Jan     100
1  Jan     120
2  Jan      90
3  Fev     110
4  Fev     130
5  Fev      95
6  Mar     105
7  Mar     125
8  Mar     100


In [13]:
# Avec identification de l'origine
df_all = pd.concat(
    [df_jan, df_fev, df_mar],
    keys=['Janvier', 'Février', 'Mars']
)
print("\nConcat avec keys (identifiant d'origine) :")
print(df_all)


Concat avec keys (identifiant d'origine) :
          mois  ventes
Janvier 0  Jan     100
        1  Jan     120
        2  Jan      90
Février 0  Fev     110
        1  Fev     130
        2  Fev      95
Mars    0  Mar     105
        1  Mar     125
        2  Mar     100


### ✍️ Exercice 6.4 : Concat de fichiers mensuels (10 min)

In [14]:
import pandas as pd
import numpy as np

# Simuler des fichiers mensuels
def creer_donnees_mois(mois, n=30):
    np.random.seed(mois)
    return pd.DataFrame({
        'date': pd.date_range(f'2024-{mois:02d}-01', periods=n, freq='D')[:n],
        'ventes': np.random.randint(100, 500, n),
        'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], n)
    })

df_jan = creer_donnees_mois(1)
df_fev = creer_donnees_mois(2)
df_mar = creer_donnees_mois(3)

print(f"Janvier : {len(df_jan)} lignes")
print(f"Février : {len(df_fev)} lignes")
print(f"Mars : {len(df_mar)} lignes")

Janvier : 30 lignes
Février : 30 lignes
Mars : 30 lignes


In [15]:
# 1. Concaténer les trois mois
df_q1 = pd.concat([df_jan, df_fev, df_mar], ignore_index=True)
print(f"\nQ1 total : {len(df_q1)} lignes")


Q1 total : 90 lignes


In [16]:
# 2. Ajouter une colonne pour identifier le mois d'origine
df_jan['mois'] = 'Janvier'
df_fev['mois'] = 'Février'
df_mar['mois'] = 'Mars'
df_q1_tagged = pd.concat([df_jan, df_fev, df_mar], ignore_index=True)

# 3. Calculer les ventes totales par mois
print("\nVentes par mois :")
print(df_q1_tagged.groupby('mois')['ventes'].sum())


Ventes par mois :
mois
Février    8229
Janvier    9730
Mars       9300
Name: ventes, dtype: int64


> 💭 **Question Socratique #3** : Vous devez combiner des données clients provenant de deux systèmes différents (CRM et e-commerce). Les deux ont une colonne "client_id" mais les ID ne sont pas les mêmes. Comment aborderiez-vous ce problème ?

Pour être rigoureux : on ne “merge” pas directement deux tables CRM et e-commerce sous prétexte qu’elles ont une colonne appelée client_id. Le nom est trompeur, la sémantique ne l’est pas.

Voici une approche méthodique et professionnelle, étape par étape.

⸻

1️⃣ Refuser la jointure naïve (point clé)

Même si les deux colonnes s’appellent client_id :
	•	ce sont deux espaces d’identifiants différents
	•	une jointure directe produirait des matchs faux ou silencieux

👉 Première règle data : un identifiant n’a de sens que dans son système.

⸻

2️⃣ Identifier une clé de rapprochement fiable

Chercher des attributs communs permettant de relier les clients :
	•	Email (le plus courant)
	•	Numéro de téléphone (attention au format)
	•	Nom + prénom + date de naissance (fragile)
	•	Adresse postale normalisée
	•	Identifiant tiers (ex : Stripe, Auth0, SSO)

⚠️ Toujours vérifier :
	•	unicité
	•	taux de null
	•	taux de collision

Exemple :

email → souvent 1–1 mais parfois 1–N (multi-comptes)


⸻

3️⃣ Construire une table de correspondance (table de mapping)

C’est la bonne pratique industrielle.

client_mapping
--------------------------------
crm_client_id | ecommerce_client_id | match_source | confidence

	•	créée une fois
	•	enrichie progressivement
	•	versionnée
	•	auditée

👉 Cette table devient la clé de vérité.

⸻

4️⃣ Mettre en place une stratégie de matching

a) Matching déterministe (prioritaire)
	•	email exact
	•	téléphone exact après normalisation
	•	ID externe commun

crm.email == ecommerce.email

b) Matching probabiliste (en dernier recours)
	•	fuzzy matching (nom, adresse)
	•	score de similarité
	•	validation humaine

⚠️ Jamais automatique sans seuil + revue

⸻

5️⃣ Normaliser avant de matcher (souvent oublié)

Exemples :
	•	emails en minuscules
	•	suppression des espaces
	•	format téléphone E.164
	•	accents / caractères spéciaux

Sinon → faux négatifs massifs.

⸻

6️⃣ Gérer les cas complexes explicitement

Cas à anticiper :
	•	1 client CRM → plusieurs comptes e-commerce
	•	comptes e-commerce sans CRM
	•	clients CRM jamais convertis
	•	clients supprimés / anonymisés (RGPD)

👉 On ne “corrige” pas ces cas, on les modélise.

⸻

8️⃣ Mesurer la qualité du rapprochement (indispensable)

Toujours produire :
	•	% de clients matchés
	•	% non matchés
	•	% matchés automatiquement vs manuellement
	•	évolution dans le temps

Sans KPI → illusion de qualité.

⸻

🧠 Conclusion claire

👉 Même nom ≠ même identifiant
👉 On ne joint pas, on réconcilie
👉 La solution passe par :
	•	compréhension métier
	•	table de mapping
	•	règles explicites
	•	métriques de qualité